<a href="https://colab.research.google.com/github/mandara-samanalee/FYP_Team_Intellicore/blob/main/notebooks/module3_activity_pattern_analysis/V1/V1_Training_Isolation_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Isolation Forest Training (with Per-Product Thresholds) - V1

This notebook trains an **Isolation Forest** anomaly detection model on genuine review history features to identify potentially suspicious review activity.

### 📌 Purpose
Instead of applying a single blanket threshold across all items, this pipeline computes **per-product anomaly thresholds**. This accounts for the fact that highly popular products have a naturally different baseline of review frequency and inter-arrival times compared to low-volume or niche products.

### 📥 Inputs
* **`df_features_train.csv`**: Genuine (non-generated) training dataset consisting of sliding-window statistical features calculated over review histories.

### 📤 Outputs
* `isolation_forest_module3_V1.pkl`: The trained `IsolationForest` estimator.
* `module3_feature_cols_V1.pkl`: List of feature columns used for inference.
* `module3_product_thresholds_V1.pkl`: A Python dictionary mapping `product_id` to its unique anomaly score threshold.
* `module3_global_threshold_V1.pkl`: A fallback threshold value applied to low-volume products.

### ⚙️ Important Constants & Thresholds
* **Feature Columns**: `['review_count', 'mean_iat', 'std_iat', 'burstiness']`
* **`contamination = 0.05`**: Initial assumed proportion of outliers in the training set.
* **`PERCENTILE = 0.05`** (5th percentile): Score cutoff used to define the boundaries of normal behavior.
* **`MIN_WINDOWS_FOR_OWN_THRESHOLD = 30`**: The minimum number of historical windows a product must have to receive a custom threshold. Products below this count default to the global fallback.

### 📊 Key Findings from Training
* **Dataset Scale**: Trained successfully on **893,238** sliding windows across **13,196** distinct products.
* **Custom vs. Fallback**: **11,867** products qualified for their own tailored threshold, while **1,329** low-volume products fell back to the global threshold.
* **Global Baseline**: The 5th percentile global threshold resolved to **0.0000** (where lower anomaly scores denote a higher likelihood of being an outlier).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

path = "/content/drive/MyDrive/df_features_train.csv"
print("Exists:", os.path.exists(path))

# If False, list what's actually in MyDrive to find the real name/location
if not os.path.exists(path):
    print("\nFiles in MyDrive:")
    for f in os.listdir("/content/drive/MyDrive"):
        if f.endswith(".csv"):
            print("  ", f)

Exists: True


In [5]:
# ============================================================
# MODULE 3 — ISOLATION FOREST TRAINING (with per-product thresholds)
# New notebook — genuine training data only
# ============================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import joblib

# ------------------------------------------------------------
# 1. Load genuine training features
# ------------------------------------------------------------
df_features_train = pd.read_csv(
    "/content/drive/MyDrive/df_features_train.csv"
)

print(f"Training windows: {len(df_features_train):,}")
print(f"Columns: {list(df_features_train.columns)}")

# ------------------------------------------------------------
# 2. Select feature columns
# Real features: review_count, mean_iat, std_iat, burstiness
# window_start/window_end are time boundaries, NOT features
# ------------------------------------------------------------
non_feature_cols = ["product_id", "window_start", "window_end",
                     "is_generated"]

feature_cols = [c for c in df_features_train.columns
                if c not in non_feature_cols]

X_train = df_features_train[feature_cols].copy()

print(f"Feature columns used ({len(feature_cols)}):")
print(feature_cols)

# ------------------------------------------------------------
# 3. Handle missing values (if any)
# ------------------------------------------------------------
if X_train.isna().sum().sum() > 0:
    X_train = X_train.fillna(0)
    print("\nNaNs filled with 0")
else:
    print("\nNo NaNs found — clean data")

# ------------------------------------------------------------
# 4. Train Isolation Forest (one global model)
# ------------------------------------------------------------
iso_forest = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination=0.05,   # starting guess — tune later on validation set
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)

print("\nTraining Isolation Forest...")
iso_forest.fit(X_train)
print("Training complete.")

# ------------------------------------------------------------
# 5. Compute anomaly scores for every training window
#    (lower score = more anomalous)
# ------------------------------------------------------------
df_features_train["anomaly_score"] = iso_forest.decision_function(X_train)

product_counts = df_features_train["product_id"].value_counts()
print(f"\nProducts with fewer than 30 windows: {(product_counts < 30).sum():,}")
print(f"Products with fewer than 10 windows: {(product_counts < 10).sum():,}")
print(f"\nWindow count distribution per product:")
print(product_counts.describe())

# ------------------------------------------------------------
# 6. Compute per-product thresholds (with global fallback)
#    Baseline differs per product, so each product gets its
#    own cutoff; low-volume products fall back to global.
# ------------------------------------------------------------
MIN_WINDOWS_FOR_OWN_THRESHOLD = 30
PERCENTILE = 0.05  # bottom 5% of scores treated as anomalous, per product

global_threshold = df_features_train["anomaly_score"].quantile(PERCENTILE)

product_thresholds = {}

for product_id, group in df_features_train.groupby("product_id"):
    if len(group) >= MIN_WINDOWS_FOR_OWN_THRESHOLD:
        product_thresholds[product_id] = group["anomaly_score"].quantile(PERCENTILE)
    else:
        product_thresholds[product_id] = global_threshold

print(f"\nProducts with own threshold: "
      f"{(product_counts >= MIN_WINDOWS_FOR_OWN_THRESHOLD).sum():,}")
print(f"Products using global fallback: "
      f"{(product_counts < MIN_WINDOWS_FOR_OWN_THRESHOLD).sum():,}")
print(f"Global fallback threshold: {global_threshold:.4f}")

# ------------------------------------------------------------
# 7. Save model, feature list, thresholds
# ------------------------------------------------------------
joblib.dump(iso_forest, "/content/drive/MyDrive/Research/models/V1/isolation_forest_module3_V1.pkl")
joblib.dump(feature_cols, "/content/drive/MyDrive/Research/models/V1/module3_feature_cols_V1.pkl")
joblib.dump(product_thresholds, "/content/drive/MyDrive/Research/models/V1/module3_product_thresholds_V1.pkl")
joblib.dump(global_threshold, "/content/drive/MyDrive/Research/models/V1/module3_global_threshold_V1.pkl")

print("\nSaved:")
print("  isolation_forest_module3_V1.pkl       — trained model")
print("  module3_feature_cols_V1.pkl           — feature column list")
print("  module3_product_thresholds_V1.pkl     — dict: product_id → threshold")
print("  module3_global_threshold_V1.pkl       — fallback for low-volume products")

Training windows: 893,238
Columns: ['product_id', 'window_start', 'window_end', 'review_count', 'mean_iat', 'std_iat', 'burstiness', 'is_generated']
Feature columns used (4):
['review_count', 'mean_iat', 'std_iat', 'burstiness']

No NaNs found — clean data

Training Isolation Forest...
Training complete.

Products with fewer than 30 windows: 1,329
Products with fewer than 10 windows: 433

Window count distribution per product:
count    13196.000000
mean        67.690058
std         24.655102
min          1.000000
25%         53.000000
50%         75.000000
75%         88.000000
max         96.000000
Name: count, dtype: float64

Products with own threshold: 11,867
Products using global fallback: 1,329
Global fallback threshold: 0.0000

Saved:
  isolation_forest_module3_V1.pkl       — trained model
  module3_feature_cols_V1.pkl           — feature column list
  module3_product_thresholds_V1.pkl     — dict: product_id → threshold
  module3_global_threshold_V1.pkl       — fallback for low-

In [6]:
# Diagnostic — understand the score distribution and that 0.0000 threshold
scores = df_features_train["anomaly_score"]

print("Anomaly score distribution:")
print(scores.describe())
print(f"\n5th percentile (= global threshold): {scores.quantile(0.05):.6f}")
print(f"Median score: {scores.median():.6f}")
print(f"How many scores are exactly 0: {(scores == 0).sum():,}")
print(f"How many scores below 0: {(scores < 0).sum():,}")
print(f"How many scores above 0: {(scores > 0).sum():,}")

Anomaly score distribution:
count    893238.000000
mean          0.181988
std           0.086902
min          -0.123662
25%           0.119606
50%           0.229913
75%           0.235218
max           0.235218
Name: anomaly_score, dtype: float64

5th percentile (= global threshold): 0.000000
Median score: 0.229913
How many scores are exactly 0: 0
How many scores below 0: 44,662
How many scores above 0: 848,576


In [7]:
import joblib
import numpy as np

product_thresholds = joblib.load("/content/drive/MyDrive/Research/models/V1/module3_product_thresholds_V1.pkl")
global_threshold   = joblib.load("/content/drive/MyDrive/Research/models/V1/module3_global_threshold_V1.pkl")

thresh_values = np.array(list(product_thresholds.values()))
print("Per-product threshold distribution:")
print(f"  min:    {thresh_values.min():.4f}")
print(f"  25%:    {np.percentile(thresh_values, 25):.4f}")
print(f"  median: {np.percentile(thresh_values, 50):.4f}")
print(f"  75%:    {np.percentile(thresh_values, 75):.4f}")
print(f"  max:    {thresh_values.max():.4f}")
print(f"  global: {global_threshold:.4f}")
print(f"\nDiffer from global by >0.01: {(np.abs(thresh_values - global_threshold) > 0.01).sum():,} / {len(thresh_values):,}")
print(f"Differ by >0.05: {(np.abs(thresh_values - global_threshold) > 0.05).sum():,}")
print(f"Equal to global (fallback ones): {(thresh_values == global_threshold).sum():,}")

Per-product threshold distribution:
  min:    -0.1116
  25%:    0.0000
  median: 0.0338
  75%:    0.0974
  max:    0.2299
  global: 0.0000

Differ from global by >0.01: 10,505 / 13,196
Differ by >0.05: 6,212
Equal to global (fallback ones): 1,329
